# GLM Rats ASD Cohort 2

Notebook wrapper for the standalone GLMM scripts. The preferred workflow here is to launch `GLM_rats_ASDcohort2_glmmTMB.py` and `GLM_rats_ASDcohort2_r2_plotting.py` from notebook cells so you can stay in Jupyter while keeping the modeling code in the runnable `.py` scripts.

## 1. Setup

Run this first. It makes imports and data paths work whether the notebook is launched from the repo root or from inside `notebooks/ASD` or `notebooks/Stakes`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "DailyMerge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

## 2. Imports

These imports support the script-driven notebook workflow. The preferred `glmmTMB` path below runs the standalone Python scripts directly, so `rpy2` is not imported here.

In [ ]:
import json
import pickle

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
conda_r_home = Path(sys.prefix) / "lib" / "R"
conda_rscript = conda_r_home / "bin" / "Rscript"
if not conda_rscript.exists():
    raise FileNotFoundError(f"Expected conda Rscript at {conda_rscript}")
os.environ["R_HOME"] = str(conda_r_home)
os.environ["PATH"] = str(conda_rscript.parent) + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

import Helpers.DataHelpers as DataHelpers
from Pipeline.group_comparison import build_group_views, load_groupcomparison_data, summarize_views

sns.set_theme(style="ticks", context="talk")
print(f"Loaded Python libraries. R_HOME={os.environ.get('R_HOME')} Rscript={conda_rscript}")

## 3. Choose Dataset

`DATASET_SELECTIONS` follows the same pattern as the group comparison, biased blocks, and learning curves notebooks. Keep `COMPARISON = "genotypes"` for the original WT/HET/HOM GLM comparison.

In [ ]:
LINES = ["CNTNAP2"]
COHORTS = ["cohort2"]

# Explicit selections can mix lines/cohorts, e.g.:
# DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("SHANK3", "cohort1")]
DATASET_SELECTIONS = [("CNTNAP2", "cohort2")]

COMPARISON = "genotypes"  # "genotypes", "datasets", "lines", "cohorts", or "custom"
SPLIT_BY = "none"         # for COMPARISON="genotypes": "none", "dataset", "line", or "cohort"
GENOTYPES = ["wt", "het", "hom"]
CUSTOM_SPECS = None

BASE_DATA_DIR = ROOT / "DataFiles"
LINE = DATASET_SELECTIONS[0][0] if DATASET_SELECTIONS else LINES[0]
COHORT = DATASET_SELECTIONS[0][1] if DATASET_SELECTIONS else COHORTS[0]
OUTPUT_STEM = "__".join(
    f"{line.lower()}_{cohort}" for line, cohort in (DATASET_SELECTIONS or [(line, cohort) for line in LINES for cohort in COHORTS])
)
OUTPUT_DIR = ROOT / "outputs" / "glm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATASET_SELECTIONS)
print(OUTPUT_DIR)

## 4. Run glmmTMB Script Pipeline

This is the preferred path. It calls the standalone `glmmTMB` fitting script, locates the newest results bundle for the selected dataset, and then runs the plotting/post-processing script on that bundle.

In [ ]:
GLMM_GROUPS = "all"      # e.g. "1", "1,2", or "all"
GLMM_OUTCOME = "choice"  # "choice" or "accuracy"
EXCLUDE_REPEAT_TRIALS = True
TRAINING_LEVEL = 16
APPLY_RT_STIM_FILTER = True
RT_SESSION_TYPE = 1
STIM_DUR_MS = 6000
SESSION_TYPES_FIRST_BLOCK_ONLY = [3, 23]
MIN_SESSION_BY_DATASET = {
    ("CNTNAP2", "cohort2"): 13,
}
COHORT_RANDOM_INTERCEPT = len(DATASET_SELECTIONS) > 1
RUN_PLOTTING = True
PLOT_BLUPS = False
PLOT_CONDITIONAL_EFFECTS = True

GLMM_SCRIPT = ROOT / "GLM_rats_ASDcohort2_glmmTMB.py"
PLOTTING_SCRIPT = ROOT / "GLM_rats_ASDcohort2_r2_plotting.py"
DATASET_KEYS = [f"{line}:{cohort}" for line, cohort in DATASET_SELECTIONS]
POOLED_RUN = len(DATASET_SELECTIONS) > 1

dataset_filter_overrides = {
    f"{line}:{cohort}": {
        "min_session": MIN_SESSION_BY_DATASET[(line, cohort)]
    }
    for line, cohort in DATASET_SELECTIONS
    if (line, cohort) in MIN_SESSION_BY_DATASET and MIN_SESSION_BY_DATASET[(line, cohort)] is not None
}

fit_cmd = [
    sys.executable,
    str(GLMM_SCRIPT),
    "--base-data-dir", str(BASE_DATA_DIR),
    "--groups", GLMM_GROUPS,
    "--outcome", GLMM_OUTCOME,
    f"--{'exclude' if EXCLUDE_REPEAT_TRIALS else 'no-exclude'}-repeat-trials",
    "--training-level", str(TRAINING_LEVEL),
    f"--{'apply' if APPLY_RT_STIM_FILTER else 'no-apply'}-rt-stim-filter",
    "--rt-session-type", str(RT_SESSION_TYPE),
    "--stim-dur-ms", str(STIM_DUR_MS),
    "--session-types-first-block-only", ",".join(str(v) for v in SESSION_TYPES_FIRST_BLOCK_ONLY),
    f"--{'cohort-random-intercept' if COHORT_RANDOM_INTERCEPT else 'no-cohort-random-intercept'}",
]
if POOLED_RUN:
    fit_cmd.extend(["--datasets", ",".join(DATASET_KEYS)])
else:
    fit_cmd.extend(["--line", LINE, "--cohort", COHORT])
    if (LINE, COHORT) in MIN_SESSION_BY_DATASET and MIN_SESSION_BY_DATASET[(LINE, COHORT)] is not None:
        fit_cmd.extend(["--min-session", str(MIN_SESSION_BY_DATASET[(LINE, COHORT)])])
if dataset_filter_overrides:
    fit_cmd.extend(["--dataset-filter-overrides", json.dumps(dataset_filter_overrides)])

print("Running:", " ".join(fit_cmd))
subprocess.run(fit_cmd, cwd=ROOT, check=True)

results_root = BASE_DATA_DIR / "runs_glmmTMB_pooled" if POOLED_RUN else (BASE_DATA_DIR / f"{LINE}_{COHORT}" / "runs_glmmTMB")
results_candidates = sorted(results_root.glob("*/glmmTMB_results_cntnap2.pkl"), key=lambda path: path.stat().st_mtime)
if not results_candidates:
    raise FileNotFoundError(f"No glmmTMB results found under {results_root}")

LATEST_GLMM_RESULTS = results_candidates[-1]
print("Latest results bundle:", LATEST_GLMM_RESULTS)

with LATEST_GLMM_RESULTS.open("rb") as f:
    latest_bundle = pickle.load(f)
latest_outcome = latest_bundle.get("outcome") or latest_bundle.get("run_config", {}).get("outcome", GLMM_OUTCOME)

if RUN_PLOTTING:
    plot_cmd = [
        sys.executable,
        str(PLOTTING_SCRIPT),
        "--results", str(LATEST_GLMM_RESULTS),
    ]
    if PLOT_BLUPS:
        plot_cmd.append("--plot-blups")
    if not PLOT_CONDITIONAL_EFFECTS:
        plot_cmd.append("--no-plot-conditional-effects")

    print("Running:", " ".join(plot_cmd))
    subprocess.run(plot_cmd, cwd=ROOT, check=True)

    figure_paths = [LATEST_GLMM_RESULTS.with_name(f"{latest_outcome}_fixed_effects_with_random.png")]
    if PLOT_BLUPS:
        figure_paths.append(LATEST_GLMM_RESULTS.with_name(f"{latest_outcome}_subject_blups.png"))
    if PLOT_CONDITIONAL_EFFECTS:
        figure_paths.append(LATEST_GLMM_RESULTS.with_name(f"{latest_outcome}_conditional_effects.png"))

    for figure_path in figure_paths:
        if figure_path.exists():
            print(f"Displaying {figure_path.name}")
            display(Image(filename=str(figure_path)))
        else:
            print(f"Figure not found: {figure_path}")

LATEST_GLMM_RESULTS

## 5. Pooled Genotype-by-Cohort Inference Model

This separate pooled model keeps genotype inside one shared GLMM so you can test genotype effects, cohort effects, and genotype-by-cohort interactions directly.

In [ ]:
INFERENCE_SCRIPT = ROOT / "GLM_rats_genotype_cohort_inference_glmmTMB.py"

if len(DATASET_SELECTIONS) < 2:
    raise ValueError("Use at least two dataset selections for the pooled genotype-by-cohort inference model.")

inference_cmd = [
    sys.executable,
    str(INFERENCE_SCRIPT),
    "--datasets", ",".join(DATASET_KEYS),
    "--base-data-dir", str(BASE_DATA_DIR),
    "--outcome", GLMM_OUTCOME,
    f"--{'exclude' if EXCLUDE_REPEAT_TRIALS else 'no-exclude'}-repeat-trials",
    "--training-level", str(TRAINING_LEVEL),
    f"--{'apply' if APPLY_RT_STIM_FILTER else 'no-apply'}-rt-stim-filter",
    "--rt-session-type", str(RT_SESSION_TYPE),
    "--stim-dur-ms", str(STIM_DUR_MS),
    "--session-types-first-block-only", ",".join(str(v) for v in SESSION_TYPES_FIRST_BLOCK_ONLY),
]
if dataset_filter_overrides:
    inference_cmd.extend(["--dataset-filter-overrides", json.dumps(dataset_filter_overrides)])

print("Running:", " ".join(inference_cmd))
subprocess.run(inference_cmd, cwd=ROOT, check=True)

inference_results_root = BASE_DATA_DIR / "runs_glmmTMB_genotype_cohort_inference"
inference_results = sorted(
    inference_results_root.glob("*/glmmTMB_genotype_cohort_results.pkl"),
    key=lambda path: path.stat().st_mtime,
)
if not inference_results:
    raise FileNotFoundError(f"No pooled inference results found under {inference_results_root}")

LATEST_GENOTYPE_COHORT_INFERENCE = inference_results[-1]
print("Latest pooled inference bundle:", LATEST_GENOTYPE_COHORT_INFERENCE)
LATEST_GENOTYPE_COHORT_INFERENCE

## 6. Legacy notebook-native lme4 workflow

The remaining cells below keep the older notebook-native `lme4::glmer` path for reference and comparison. For day-to-day use in this repo, prefer the script-driven section above.

That is why the trial filtering appears after the new `glmmTMB` runner: the standalone script already loads and filters trials internally, while the cells below rebuild that logic only for the legacy in-notebook path.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import Formula, pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

lme4 = importr("lme4")
stats = importr("stats")
MuMIn = importr("MuMIn")

print(f"Loaded legacy R libraries. R_HOME={os.environ.get('R_HOME')} Rscript={conda_rscript}")

## Legacy 1. Load And Filter Trials

This loads cohort data plus animal metadata through the shared dataset loader, then follows the MATLAB/script filtering: no repeated trials, training level 16, RT sessions or 6000 ms stimuli, session >= 13, response recoded from `-1/1` to `0/1`, and fixation time converted to seconds.

In [ ]:
data_bundle = load_groupcomparison_data(
    lines=LINES,
    cohorts=COHORTS,
    dataset_selections=DATASET_SELECTIONS,
    base_dir=BASE_DATA_DIR,
    require_meta=True,
)

df = data_bundle["df_plot"].copy()
df = DataHelpers.prepare_data(df, session_col="session", trial_col="trial")

df = df[df["trial_is_repeat"] == False].copy()
df = df[df["training_level"] == 16].copy()

sess = pd.to_numeric(df["session_type"], errors="coerce")
sd = pd.to_numeric(df["stim_dur"], errors="coerce")
df = df[(sess == 1) | (sd == 6000)].copy()

out = df.copy()
out["animal"] = out["animal"].astype(str).str.strip()
out["subject"] = out["animal"]
out["session_id"] = out["animal"] + "_S" + out["session"].astype(int).astype(str).str.zfill(3)
out["Response"] = out["response_poke"]
out["Response"] = (out["Response"] + 1) / 2
out["fix_time"] = out["fix_time"] / 1000

out["ABLc"] = out["ABL"].astype("category")
out["subject"] = out["subject"].astype("category")
out["repeated_trial"] = out["repeated_trial"].astype("category")
out = out[out["session"] >= 13].copy()

print(f"Loaded selections: {data_bundle['selections']}")
print(f"Usable datasets: {data_bundle['usable_dataset_names']}")
print(f"Filtered rows before session cutoff: {len(df):,}")
print(f"After session >= 13: {len(out):,} rows, {out['subject'].nunique()} subjects")
out[["subject", "session", "trial", "Response", "ABL", "ILD", "fix_time", "success"]].head()

## Legacy 2. Build Genotype Views

This is the same view-building pattern used by the other notebooks. Animal-to-genotype assignment comes from `sex_gen.csv` metadata loaded with the cohort, not from manual subject lists.

In [ ]:
views = build_group_views(
    out,
    comparison=COMPARISON,
    split_by=SPLIT_BY,
    genotypes=GENOTYPES,
    lines=LINES,
    cohorts=COHORTS,
    custom_specs=CUSTOM_SPECS,
)

out_type = [view.selector(out).copy() for view in views]
group_names = [view.name.upper() for view in views]

summary = summarize_views(out, views)
summary["prepared_group_label"] = group_names
summary

## Legacy 3. Configure Predictors

Use these switches to include current-trial and previous-trial predictor groups. The predictor groups are also used later for the reduced-model R2 drop analysis.

In [ ]:
response_var = "Response"

USE_CURRENT = True
USE_PREVIOUS = True

preds_grouped = []
var_group_names = []

if USE_CURRENT:
    var_group_names.append("current")
    preds_grouped.append([
        "ABL",
        "ILD",
        "fix_time_long",
        "ILD:ABL",
        "trial",
        "ILD:trial",
        "ILD:fix_time_long",
        "trial:fix_time_long",
    ])

if USE_PREVIOUS:
    var_group_names.append("pre")
    preds_grouped.append([
        "Pre_choice",
        "Pre_success",
        "ILD:Pre_success",
        "Pre_success:Pre_choice",
    ])

all_fixed_predictors = sum(preds_grouped, [])
pred_to_zscore = all_fixed_predictors

subject_random_slopes = [
    "ABL",
    "ILD",
    "ILD:ABL",
    "trial",
    "ILD:trial",
    "Pre_choice",
    "Pre_success",
    "ILD:Pre_success",
    "Pre_success:Pre_choice",
]
sessionID_random_slopes = "ILD:ABL"

all_fixed_predictors

## Legacy 4. Helper Functions

These helpers build the per-subject/per-session predictors, z-score predictors within each session, and convert the model table into an R-friendly dataframe.

In [ ]:
def zscore_fun(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / series.std(ddof=1)


def join_slopes(slopes) -> str:
    if slopes is None:
        return ""
    if isinstance(slopes, str):
        return slopes
    slopes = [s for s in slopes if s]
    if not slopes:
        return ""
    if all(isinstance(s, str) and len(s) == 1 for s in slopes):
        return "".join(slopes)
    return " + ".join(slopes)


def build_group_model_table(df_group: pd.DataFrame) -> pd.DataFrame:
    df_out = []
    subjects = df_group["subject"].unique()

    for subj_i, subj in enumerate(subjects, start=1):
        subj_df = df_group[df_group["subject"] == subj].copy()
        sess_unique = np.sort(subj_df["session"].unique())
        sess_map = {s: i + 1 for i, s in enumerate(sess_unique)}
        subj_df["session"] = subj_df["session"].map(sess_map)

        for sess in np.sort(subj_df["session"].unique()):
            s_df = subj_df[subj_df["session"] == sess].copy()
            s_df = s_df.sort_values("trial").reset_index(drop=True)

            s_df["trial"] = s_df["trial"] - s_df["trial"].mean()
            s_df["Pre_ILD"] = s_df["ILD"].shift(1).fillna(0)
            s_df["Pre_ABL"] = s_df["ABL"].shift(1).fillna(0)
            s_df["Pre_choice"] = s_df["Response"].shift(1).fillna(0)
            s_df["Pre_success"] = s_df["success"].shift(1)

            s_df["ILD:ABL"] = s_df["ILD"] * s_df["ABL"]
            s_df["ILD:trial"] = s_df["ILD"] * s_df["trial"]
            s_df["fix_time_long"] = (s_df["fix_time"] > s_df["fix_time"].median()).astype(float)
            s_df["ILD:fix_time_long"] = s_df["ILD"] * s_df["fix_time_long"]
            s_df["trial:fix_time_long"] = s_df["trial"] * s_df["fix_time_long"]
            s_df["ILD:Pre_success"] = s_df["ILD"] * s_df["Pre_success"]
            s_df["Pre_success:Pre_choice"] = s_df["Pre_success"] * s_df["Pre_choice"]

            for col in pred_to_zscore:
                if col in s_df.columns:
                    s_df[col] = zscore_fun(s_df[col])

            df_out.append(s_df)

    df_out = pd.concat(df_out, ignore_index=True)
    df_out = df_out.dropna(subset=[response_var])
    num_cols = df_out.select_dtypes(include=[np.number]).columns
    df_out[num_cols] = df_out[num_cols].fillna(0)
    return df_out


def prep_for_r_glmm(df: pd.DataFrame) -> pd.DataFrame:
    fixed = all_fixed_predictors
    random_groups = ["session_id", "subject"]
    response = "Response"
    keep = [response] + fixed + random_groups
    df = df.loc[:, [c for c in keep if c in df.columns]].copy()

    df[response] = df[response].astype(float).round().astype(int)
    for c in fixed:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    for g in random_groups:
        df[g] = df[g].astype(str)

    return df.dropna(subset=[response] + fixed + random_groups)


def glmer_binomial(formula_str: str, df_clean: pd.DataFrame):
    with localconverter(ro.default_converter + pandas2ri.converter):
        df_r = ro.conversion.py2rpy(df_clean)
    return lme4.glmer(Formula(formula_str), data=df_r, family=stats.binomial(link="logit"))

## Legacy 5. Prepare Once

Rerun this cell if you change filtering or predictor configuration. You do not need to rerun it just to change plotting.

In [ ]:
out_glm3 = []

for group_name, df_group in zip(group_names, out_type):
    print(f"Preparing {group_name}: {len(df_group):,} rows, {df_group['subject'].nunique()} subjects")
    df_out = build_group_model_table(df_group)
    df_out = df_out[df_out["trial_is_repeat"] == False].copy()
    out_glm3.append(df_out)
    print(f"  ready: {len(df_out):,} rows")

pd.DataFrame({"group": group_names, "prepared_rows": [len(d) for d in out_glm3]})

## Legacy 6. Fit Full GLMMs

This is the slow step. It fits one model per genotype group with diagonal random-effect covariance using `||`, matching the original script.

In [ ]:
mdle_all = []

fixed_part = "1 + " + " + ".join(all_fixed_predictors)
subject_slopes_str = join_slopes(subject_random_slopes)
sessionID_slopes_str = join_slopes(sessionID_random_slopes)
random_part = (
    f"(1 + {sessionID_slopes_str} || session_id) + "
    f"(1 + {subject_slopes_str} || subject)"
)
formula_str = f"Response ~ {fixed_part} + {random_part}"
print(formula_str)

for group_name, df_out in zip(group_names, out_glm3):
    print(f"\nFitting {group_name} GLMM...")
    df_clean = prep_for_r_glmm(df_out)
    print(f"Dataframe ready: {len(df_clean):,} rows, {df_clean['subject'].nunique()} subjects")
    glmer_fit = glmer_binomial(formula_str, df_clean)
    mdle_all.append([glmer_fit])
    print(f"Finished {group_name}.")

## Legacy 7. Save Fitted Models

The saved pickle contains the R model objects and prepared Python tables, so you can reuse them without rebuilding predictors.

In [ ]:
model_save_path = OUTPUT_DIR / f"glmm_results_{OUTPUT_STEM}.pkl"

to_save = {
    "mdle_all": mdle_all,
    "out_glm3": out_glm3,
    "group_names": group_names,
    "all_fixed_predictors": all_fixed_predictors,
    "preds_grouped": preds_grouped,
    "var_group_names": var_group_names,
    "subject_random_slopes": subject_random_slopes,
    "sessionID_random_slopes": sessionID_random_slopes,
}

with model_save_path.open("wb") as f:
    pickle.dump(to_save, f)

print(model_save_path)

## Legacy 8. Compute R2 Drops

This fits a full model and reduced models with simple random intercepts, then stores the fractional marginal R2 drop for each predictor group.

In [ ]:
frac_R2_all = []
r2_func = ro.r("MuMIn::r.squaredGLMM")

for group_name, df_out in zip(group_names, out_glm3):
    print(f"\nComputing R2 drops for {group_name}...")
    df_clean = prep_for_r_glmm(df_out)

    fixed_full_str = "1 + " + " + ".join(all_fixed_predictors)
    random_full_r2 = "(1 | session_id) + (1 | subject)"
    formula_full_r2_str = f"Response ~ {fixed_full_str} + {random_full_r2}"
    full_model_r2 = glmer_binomial(formula_full_r2_str, df_clean)

    R2_full = float(r2_func(full_model_r2)[0])
    R2_vec = [R2_full]
    print(f"  marginal R2 full: {R2_full:.4f}")

    for preds_to_remove, var_group_name in zip(preds_grouped, var_group_names):
        kept_fixed = [p for p in all_fixed_predictors if p not in preds_to_remove]
        kept_fixed_core = join_slopes(kept_fixed)
        kept_fixed_str = "1 + " + kept_fixed_core if kept_fixed_core else "1"
        formula_red_r2_str = f"Response ~ {kept_fixed_str} + {random_full_r2}"
        reduced_fit_r2 = glmer_binomial(formula_red_r2_str, df_clean)
        R2_reduced = float(r2_func(reduced_fit_r2)[0])
        R2_vec.append(R2_reduced)
        print(f"  drop {var_group_name}: reduced R2={R2_reduced:.4f}")

    frac_R2 = [(R2_vec[0] - r) / R2_vec[0] for r in R2_vec[1:]]
    frac_R2_all.append(frac_R2)

print("Done.")

## Legacy 9. Extract Fixed Effects

Builds a compact Python structure with fixed-effect beta estimates, standard errors, t-values, and R2 fractions for plotting.

In [ ]:
fixef = ro.r["fixef"]
vcov = ro.r["vcov"]
diag = ro.r["diag"]
sqrt = ro.r["sqrt"]

model_results = []

for group_name, fit_list, frac_R2 in zip(group_names, mdle_all, frac_R2_all):
    fitres = fit_list[0]
    betas_R = fixef(fitres)
    betas = np.array(betas_R)
    names = list(betas_R.names)
    se = np.array(sqrt(diag(vcov(fitres))))
    tvals = betas / se

    model_results.append({
        "group": group_name,
        "betas": betas,
        "se": se,
        "t": tvals,
        "names": names,
        "frac_R2": frac_R2,
    })

pd.DataFrame(
    [
        {"group": res["group"], "n_terms": len(res["names"]), "r2_groups": len(res["frac_R2"])}
        for res in model_results
    ]
)

## Legacy 10. Save Extended Results

In [ ]:
extended_save_path = OUTPUT_DIR / f"glmm_results_{OUTPUT_STEM}_full.pkl"

extra = {
    "mdle_all": mdle_all,
    "out_glm3": out_glm3,
    "frac_R2_all": frac_R2_all,
    "model_results": model_results,
    "group_names": group_names,
    "all_fixed_predictors": all_fixed_predictors,
    "preds_grouped": preds_grouped,
    "var_group_names": var_group_names,
}

with extended_save_path.open("wb") as f:
    pickle.dump(extra, f)

print(extended_save_path)

## Legacy 11. Plot Results

Rerun this cell after changing colors, labels, or figure size. It does not refit models.

In [ ]:
colors = sns.color_palette("Set1", len(model_results))
common_betanames = model_results[0]["names"]
n_pred = len(common_betanames)
y_positions = np.arange(n_pred)
jitter_offsets = np.linspace(-0.15, 0.15, len(model_results))
jitter_offsets_var = np.linspace(-0.015, 0.015, len(model_results))

fig, axes = plt.subplots(1, 3, figsize=(18, 10))
plt.subplots_adjust(wspace=0.3)

ax = axes[0]
ax.axvline(0, color="k", ls="--")
for gi, res in enumerate(model_results):
    for i in range(n_pred):
        ax.errorbar(
            res["betas"][i],
            y_positions[i] + jitter_offsets[gi],
            xerr=res["se"][i],
            fmt="o",
            color=colors[gi],
            label=res["group"] if i == 0 else "",
        )
ax.set_yticks(y_positions)
ax.set_yticklabels(common_betanames)
ax.set_xlabel("Beta +/- SE")
ax.set_title("Fixed Effects")
ax.invert_yaxis()
ax.legend(frameon=False)

ax = axes[1]
ax.axvline(0, color="k", ls="--")
ax.axvline(2, color="k", ls=":")
ax.axvline(-2, color="k", ls=":")
for gi, res in enumerate(model_results):
    for i in range(n_pred):
        ax.plot(res["t"][i], y_positions[i] + jitter_offsets[gi], "o", color=colors[gi])
ax.set_yticks(y_positions)
ax.set_yticklabels(common_betanames)
ax.set_xlabel("t-statistic")
ax.set_title("t-values")
ax.invert_yaxis()

group_x = np.arange(len(var_group_names))
ax = axes[2]
for gi, res in enumerate(model_results):
    vals = np.array(res["frac_R2"])
    for j, val in enumerate(vals):
        ax.plot(
            val,
            group_x[j] + jitter_offsets_var[gi],
            "o-",
            color=colors[gi],
            label=res["group"] if j == 0 else "",
        )
ax.set_yticks(group_x)
ax.set_yticklabels(var_group_names)
ax.set_xlabel("Fraction of R2")
ax.set_title("Variance Explained by Predictor Group")
ax.invert_yaxis()

sns.despine(fig=fig)
plt.tight_layout()
plt.show()